# KG1 NVIDIA Nemotron - Pipeline TOP 1 (v30)
## 3 Fases: SFT Perfected → CoT Distillation → GRPO RL

**Melhor score anterior**: 0.68 (v8, r32/a16, 2000ex, SEM CoT)
**TOP 1 público**: 0.82
**Meta**: ≥ 0.80

### Regras de Ouro (NUNCA violar):
1. `alpha=16, rank=32` (ratio 0.5) — TODA tentativa com alpha≥32 FALHOU
2. SEM CoT fabricado — toda vez que usamos CoT falso, score CAIU
3. SEM MoE patch manual — v8 NÃO usou e scorou 0.68
4. SEM enable_thinking no TREINO — o eval do Kaggle usa, mas o treino NÃO
5. Prompt format EXATO do v8: `prompt + \"\\nPut your final answer inside \\\\boxed{}.\"` → `\\boxed{answer}`

In [ ]:
#@title 🔧 CELL 1: Setup + Config
#@markdown ### Selecione a Fase:
PHASE = 1  #@param [1, 2, 3, 4, 5] {type:"integer"}
#@markdown ### Opções:
AUTO_SUBMIT = True  #@param {type:"boolean"}
FRESH_LORA = True  #@param {type:"boolean"}

import subprocess, sys, os, json, time, random, zipfile, shutil
from datetime import datetime, timezone
from collections import Counter
from pathlib import Path

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
print("=== Installing dependencies ===")
def pip_install(*pkgs):
    for pkg in pkgs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--root-user-action=ignore", pkg],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  ✓ {pkg}")

pip_install("peft>=0.17.0", "datasets", "accelerate", "trl>=0.17.0",
            "huggingface_hub", "kaggle", "pandas",
            "vllm>=0.18.0")  # NONUPLE: peft 0.17 = target_parameters MoE; vllm 0.18 = CVE-2026-27893 fix

# Install mamba-ssm (critical for Nemotron hybrid arch)
# BUGFIX v42.1: Colab Python 3.12 needs --no-build-isolation + visible stderr
print("  Installing mamba-ssm (takes ~3min on Colab)...")
try:
    import mamba_ssm
    print(f"  ✓ mamba-ssm {mamba_ssm.__version__} (cached)")
except ImportError:
    # Strategy 1: pinned versions with --no-build-isolation (uses existing torch)
    print("  Strategy 1: pinned versions + --no-build-isolation")
    r1 = subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-build-isolation", "-q", "--root-user-action=ignore",
         "causal-conv1d==1.5.0.post8", "mamba-ssm==2.2.4"],
        capture_output=True, text=True, timeout=900)
    if r1.returncode != 0:
        print(f"  Strategy 1 FAILED: {r1.stderr[-500:]}")
        # Strategy 2: latest version, no pin
        print("  Strategy 2: latest version, no pin")
        r2 = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--no-build-isolation", "--root-user-action=ignore",
             "causal-conv1d", "mamba-ssm"],
            capture_output=True, text=True, timeout=1200)
        if r2.returncode != 0:
            print(f"  Strategy 2 FAILED: {r2.stderr[-1000:]}")
            # Strategy 3: fallback to Python-only Mamba (slower but works)
            print("  ⚠️ mamba-ssm unavailable - using HF Python fallback (slower)")
        else:
            print("  ✓ Strategy 2 succeeded")
    else:
        print("  ✓ Strategy 1 succeeded")
    try:
        import mamba_ssm
        print(f"  ✓ mamba-ssm {mamba_ssm.__version__}")
    except ImportError:
        print("  ⚠️ mamba-ssm not importable - HF will use Python Mamba fallback")

import torch
import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download, snapshot_download

# ============================================================
# AUTHENTICATION
# ============================================================
print("\n=== Authentication ===")

# HuggingFace
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_KEY")
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
    print("  ✓ Colab secrets loaded")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN", os.environ.get("HF_KEY", ""))
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "felipe1983")
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
    print("  ✓ Environment variables loaded")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("  ✓ HuggingFace authenticated")

# Kaggle
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("  ✓ Kaggle credentials configured")

# ============================================================
# GPU INFO
# ============================================================
print(f"\n=== System ===")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name} ({gpu_mem:.0f} GB)")
    if gpu_mem < 40:
        print("  ⚠️ AVISO: GPU com < 40GB. Nemotron 30B precisa de ~60GB em BF16.")
        print("  Selecione runtime H100/A100 no Colab!")

# ============================================================
# TRITON PATCH
# ============================================================
try:
    ptxas_src = "/usr/local/cuda-12.8/bin/ptxas"
    if not os.path.exists(ptxas_src):
        ptxas_src = "/usr/local/cuda/bin/ptxas"
    if os.path.exists(ptxas_src):
        target = os.path.join(os.path.dirname(shutil.which("python") or "/usr/bin/python"), "ptxas")
        if not os.path.exists(target):
            shutil.copy2(ptxas_src, target)
            print(f"  ✓ Triton ptxas patched")
except:
    pass

api = HfApi()

# ============================================================
# PHASE-DEPENDENT CONFIG
# ============================================================
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
DATA_REPO = "felipesp1983/kg1-nemotron-training"

PHASE_CONFIGS = {
    1: {  # v41: v30 EXATO (PROVEN 0.68) + filtro verified_wrong
        "name": "v41-sft-clean",
        "output_repo": "felipesp1983/kg1-nemotron-lora-v41-clean",
        "n_examples": 5000,     # PROVEN v30 actual (was 9500 in notebook but 5000 in run)
        "n_epochs": 2,          # PROVEN v30 actual (was 3 in notebook but 2 in run)
        "learning_rate": 5e-5,
        "max_length": 1024,     # EXATO v30/v8
        "grad_accum": 8,        # EXATO v30/v8
        "warmup_ratio": 0.05,   # EXATO v30/v8
        "use_thinking": False,  # EXATO v30/v8 — NÃO usar thinking no treino
        "use_cot": False,       # EXATO v30/v8 — SEM CoT
        "warm_start_repo": None,  # Fresh LoRA
        "submit_steps": [200, 300, 400, 500, 600, 800, 1000],  # focus on v30 sweet spot ~400
        "eq_oversample": 2.0,   # PROVEN v30 value (v40 used 3 = BAD)
    },
    2: {  # v31: CoT genuíno via teacher distillation
        "name": "v31-cot-distill",
        "output_repo": "felipesp1983/kg1-nemotron-lora-v31-cot",
        "n_examples": 7000,
        "n_epochs": 2,
        "learning_rate": 2e-5,  # Menor (warm start)
        "max_length": 2048,     # Espaço para <think>
        "grad_accum": 8,
        "warmup_ratio": 0.03,
        "use_thinking": True,
        "use_cot": True,        # 75% CoT + 25% direto
        "warm_start_repo": "felipesp1983/kg1-nemotron-lora-v41-clean",
        "submit_steps": [200, 400, 600, 800],
        "eq_oversample": 3.0,   # 3x eq_transform (família mais fraca)
    },
    3: {  # v32: GRPO RL
        "name": "v32-grpo-rl",
        "output_repo": "felipesp1983/kg1-nemotron-lora-v32-grpo",
        "n_examples": 2000,     # Menor dataset para RL
        "n_epochs": 2,
        "learning_rate": 1e-6,  # Muito conservador para RL
        "max_length": 2048,
        "grad_accum": 4,
        "warmup_ratio": 0.05,
        "use_thinking": True,
        "use_cot": False,
        "warm_start_repo": "felipesp1983/kg1-nemotron-lora-v31-cot",
        "submit_steps": [100, 200, 300, 400],
        "eq_oversample": 3.0,
    },
    4: {  # v42 NONUPLE: v41 base + freeze_moe_router + checkpoint averaging
        "name": "v42-nonuple",
        "output_repo": "felipesp1983/kg1-nemotron-lora-v42-nonuple",
        "n_examples": 5000,     # Same as v41 PROVEN baseline
        "n_epochs": 2,          # Same as v41 PROVEN baseline
        "learning_rate": 5e-5,  # Same as v41 PROVEN baseline
        "max_length": 1024,     # Same as v41 PROVEN baseline
        "grad_accum": 8,        # Same as v41 PROVEN baseline
        "warmup_ratio": 0.05,   # Same as v41 PROVEN baseline
        "use_thinking": False,  # Same as v41 PROVEN baseline
        "use_cot": False,       # Same as v41 PROVEN baseline
        "warm_start_repo": None,
        # NONUPLE additions:
        "submit_steps": [200, 300, 400, 500, 600, 800, 1000],  # same as v41
        "eq_oversample": 2.0,
        "freeze_moe_router": True,    # NONUPLE: Aman Atar (168 votes) - prevent router collapse
        "exclude_out_proj": False,    # NONUPLE: NVIDIA NeMo YAML - LoRA on Mamba out_proj broken (TEST FALSE first - same as v41)
        "checkpoint_averaging": True, # NONUPLE: AIMO-2 trick - average last 4 checkpoints
        "skip_pretrain_smoke": False, # NONUPLE: enforce mandatory smoke test (feedback rule)
        "skip_prescore_gate": False,  # NONUPLE: enforce pre-score gate (feedback rule)
        "prescore_min_threshold": 0.60,  # gate: only train if smoke prescore >= 0.60
    },
    5: {  # v43 DECUPLE: v42 Phase 4 + livctr hyperparams (LB 0.74 confirmed) + solver CoT data
        "name": "v43-decuple-livctr",
        "output_repo": "felipesp1983/kg1-nemotron-lora-v43-decuple",
        # livctr exp41 hyperparams (LB 0.74 CONFIRMED, github.com/livctr/nvidia-nemotron-kaggle):
        "n_examples": 5000,     # total, distributed per family (not flat)
        "n_epochs": 1,          # livctr exp41: 1 epoch only
        "learning_rate": 1e-4,  # livctr exp41 (vs v41/v42 Phase 4: 5e-5)
        "max_length": 1536,     # livctr exp41 (vs v41: 1024)
        "grad_accum": 16,       # effective bs=16 (kbsooo v1 matches)
        "warmup_ratio": 0.05,
        "use_thinking": True,   # solver CoTs already contain <think> tags
        "use_cot": False,       # BUG FIX: originals use format_example_direct (solver examples keep real CoTs via messages)
        "warm_start_repo": None,  # fresh LoRA (adapter chaining optional in separate run)
        # NONUPLE + DECUPLE additions:
        "submit_steps": [100, 200, 300, 400],  # 4 submits (leaves 1 Kaggle daily slot for averaged)
        "eq_oversample": 2.0,
        "freeze_moe_router": True,     # NONUPLE: Aman Atar - prevent router collapse
        "exclude_out_proj": False,     # keep baseline (test True in follow-up)
        "checkpoint_averaging": True,  # NONUPLE: AIMO-2 averaging (+1-2 pts free)
        "skip_pretrain_smoke": False,  # NONUPLE: enforce smoke test
        "skip_prescore_gate": False,   # NONUPLE: enforce pre-score gate
        "prescore_min_threshold": 0.60,
        # DECUPLE NEW:
        "use_solver_augmented_data": True,  # load data/solver_augmented_train.jsonl in Cell 2
        "solver_augmented_ratio": 0.70,     # 70% solver CoTs + 30% original train.csv
        "source_recipe": "livctr_exp41_hybrid",
    },
}

CFG = PHASE_CONFIGS[PHASE]
OUTPUT_REPO = CFG["output_repo"]
OUTPUT_DIR = f"/tmp/kg1_output/{CFG['name']}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# FIXED LoRA config — NUNCA mudar estes valores
LORA_RANK = 32
LORA_ALPHA = 16       # ratio 0.5 — PROVEN
LORA_DROPOUT = 0.05

print(f"\n{'='*60}")
print(f"  PHASE {PHASE}: {CFG['name']}")
print(f"  Examples: {CFG['n_examples']} | Epochs: {CFG['n_epochs']}")
print(f"  LR: {CFG['learning_rate']} | MaxLen: {CFG['max_length']}")
print(f"  LoRA: r={LORA_RANK} α={LORA_ALPHA} (ratio={LORA_ALPHA/LORA_RANK})")
print(f"  Thinking: {CFG['use_thinking']} | CoT: {CFG['use_cot']}")
print(f"  Warm start: {CFG['warm_start_repo'] or 'Fresh LoRA'}")
print(f"  Output: {OUTPUT_REPO}")
print(f"  Auto-submit steps: {CFG['submit_steps']}")
print(f"{'='*60}")
# ============================================================
# NONUPLE FLAGS (Phase 4 only)
# ============================================================
FREEZE_MOE_ROUTER = CFG.get("freeze_moe_router", False)
EXCLUDE_OUT_PROJ = CFG.get("exclude_out_proj", False)
CHECKPOINT_AVERAGING = CFG.get("checkpoint_averaging", False)
SKIP_PRETRAIN_SMOKE = CFG.get("skip_pretrain_smoke", True)  # default skip for v41 backward compat
SKIP_PRESCORE_GATE = CFG.get("skip_prescore_gate", True)
PRESCORE_MIN = CFG.get("prescore_min_threshold", 0.60)

if PHASE == 4:
    print(f"\n{'='*60}")
    print(f"  NONUPLE FLAGS (Phase 4)")
    print(f"  freeze_moe_router: {FREEZE_MOE_ROUTER}")
    print(f"  exclude_out_proj: {EXCLUDE_OUT_PROJ}")
    print(f"  checkpoint_averaging: {CHECKPOINT_AVERAGING}")
    print(f"  pretrain smoke test: {not SKIP_PRETRAIN_SMOKE}")
    print(f"  pre-score gate: {not SKIP_PRESCORE_GATE} (min={PRESCORE_MIN})")
    print(f"{'='*60}")


# ============================================================
# DECUPLE FLAGS (Phase 5 only)
# ============================================================
USE_SOLVER_AUGMENTED_DATA = CFG.get("use_solver_augmented_data", False)
SOLVER_AUGMENTED_RATIO = CFG.get("solver_augmented_ratio", 0.0)
SOURCE_RECIPE = CFG.get("source_recipe", "v41_proven")

if PHASE == 5:
    print(f"\n{'='*60}")
    print(f"  DECUPLE FLAGS (Phase 5) — livctr LB 0.74 hybrid")
    print(f"  source_recipe: {SOURCE_RECIPE}")
    print(f"  use_solver_augmented_data: {USE_SOLVER_AUGMENTED_DATA}")
    print(f"  solver_augmented_ratio: {SOLVER_AUGMENTED_RATIO}")
    print(f"  learning_rate: {CFG['learning_rate']} (livctr exp41: 1e-4)")
    print(f"  max_length: {CFG['max_length']} (livctr exp41: 1536)")
    print(f"  n_epochs: {CFG['n_epochs']} (livctr exp41: 1)")
    print(f"  grad_accum: {CFG['grad_accum']} (effective bs={CFG['grad_accum']})")
    print(f"{'='*60}")

print("\n✅ CELL 1 COMPLETE")

In [ ]:
#@title 📊 CELL 2: Load & Prepare Data

# ============================================================
# DOWNLOAD DATA
# ============================================================
print("=== Downloading training data ===")

# ============================================================
# DECUPLE: LOAD SOLVER-AUGMENTED DATA (Phase 5 only)
# ============================================================
solver_aug_records = []
if USE_SOLVER_AUGMENTED_DATA:
    print("\n=== DECUPLE: Loading solver-augmented data ===")
    # Try local path first, then HF repo fallback
    solver_aug_path = None
    for candidate in [
        "data/solver_augmented_train.jsonl",
        "/tmp/kg1_data/data/solver_augmented_train.jsonl",
        "/content/kg1-nvidia/data/solver_augmented_train.jsonl",
    ]:
        if os.path.exists(candidate):
            solver_aug_path = candidate
            break

    if solver_aug_path is None:
        # Try to download from HF (Felipe needs to have uploaded it)
        try:
            hf_hub_download(
                repo_id=DATA_REPO,
                filename="solver_augmented_train.jsonl",
                local_dir="/tmp/kg1_data/",
                repo_type="dataset",
            )
            solver_aug_path = "/tmp/kg1_data/solver_augmented_train.jsonl"
        except Exception as e:
            print(f"  ⚠️  Could not load solver_augmented_train.jsonl: {e}")
            print(f"  ⚠️  Falling back to original training data only")

    if solver_aug_path:
        with open(solver_aug_path, encoding="utf-8") as f:
            for line in f:
                solver_aug_records.append(json.loads(line))
        print(f"  ✓ Loaded {len(solver_aug_records)} solver-augmented records")
        # Stats per family
        from collections import Counter as _Counter
        fam_counts = _Counter(r["family"] for r in solver_aug_records)
        for fam, cnt in sorted(fam_counts.items()):
            print(f"    {fam}: {cnt}")

hf_hub_download(repo_id=DATA_REPO, filename="data/train.csv", local_dir="/tmp/kg1_data")
train_df = pd.read_csv("/tmp/kg1_data/data/train.csv")
print(f"Official data: {len(train_df)} rows")

# === v41: Load wrong_ids but DO NOT filter train_df yet ===
# CRITICAL: applying filter BEFORE sampling causes seed shift (10.1% data variance vs v30).
# We load wrong_ids here and apply filter AFTER shuffle+truncate (cell 2 end).
# This guarantees v41 = v30 EXACT 5000 - 89 wrong = 4911 examples (mathematical subset).
wrong_ids = set()
try:
    hf_hub_download(repo_id=DATA_REPO, filename="train_verified.csv",
                    local_dir="/tmp/kg1_data", repo_type="dataset")
    verified_path = "/tmp/kg1_data/train_verified.csv"
    verified_df = pd.read_csv(verified_path)
    wrong_ids = set(verified_df[verified_df["status"] == "verified_wrong"]["id"])
    print(f"  v41 wrong_ids loaded: {len(wrong_ids)} (will filter AFTER sampling)")
except Exception as e:
    print(f"  WARNING: train_verified.csv not found ({e}), no wrong filter applied")

# Try to load solver results for quality filtering
solver_results = None
try:
    hf_hub_download(repo_id=DATA_REPO, filename="data/solver_results.json", local_dir="/tmp/kg1_data")
    with open("/tmp/kg1_data/data/solver_results.json") as f:
        solver_results = json.load(f)
    correct_ids = {r.get("id", r.get("index", -1)) for r in solver_results if r.get("correct", False)}
    print(f"Solver-correct: {len(correct_ids)} / {len(solver_results)}")
except Exception as e:
    print(f"No solver results: {e}")
    correct_ids = None

# ============================================================
# CLASSIFY FAMILIES (same as v30)
# ============================================================
def classify(prompt):
    p = prompt.lower()
    if "bit manipulation" in p: return "bit"
    if "gravitational" in p or "gravity" in p: return "grav"
    if "unit conversion" in p or "measurement" in p: return "unit"
    if "numeral" in p: return "num"
    if "encryption" in p or "cipher" in p: return "enc"
    if "transformation" in p: return "eq"
    return "other"

train_df["family"] = train_df["prompt"].apply(classify)
train_df["ans_len"] = train_df["answer"].astype(str).str.len()

# ============================================================
# FILTER: solver-correct + answer length ≤ 24 (same as v30)
# ============================================================
filtered_df = train_df.copy()

# Filter solver-correct if available
if correct_ids:
    filtered_df = filtered_df[filtered_df.index.isin(correct_ids)]
    print(f"After solver filter: {len(filtered_df)}")

# Filter answer length (Kaggle max_answer_length=24)
filtered_df = filtered_df[filtered_df["ans_len"] <= 24]
print(f"After answer length filter (≤24): {len(filtered_df)}")

print(f"\nFiltered family distribution (same as v30):")
print(filtered_df["family"].value_counts().to_string())

# ============================================================
# PREPARE EXAMPLES (PHASE-DEPENDENT, IDENTICAL TO v30)
# ============================================================
random.seed(42)

def format_example_direct(row):
    """v8 format: NO CoT, NO thinking, JUST \\boxed{answer}"""
    return {
        "messages": [
            {"role": "user", "content": row["prompt"] + "\nPut your final answer inside \\boxed{}." },
            {"role": "assistant", "content": f"\\boxed{{{row['answer']}}}"},
        ]
    }

def format_example_cot(row, reasoning=""):
    """Phase 2 format: <think>reasoning</think>\\boxed{answer}"""
    if not reasoning:
        reasoning = generate_reasoning(row)
    return {
        "messages": [
            {"role": "user", "content": row["prompt"] + "\nPut your final answer inside \\boxed{}." },
            {"role": "assistant", "content": f"<think>\n{reasoning}\n</think>\n\\boxed{{{row['answer']}}}"},
        ]
    }

def generate_reasoning(row):
    """Generate minimal genuine reasoning based on family."""
    fam = row["family"]
    ans = str(row["answer"])
    if fam == "grav":
        return f"I need to find the gravitational constant g from the given time-distance pairs using d = 0.5*g*t^2. Analyzing the examples and solving for g, then computing the result. The answer is {ans}."
    elif fam == "unit":
        return f"I need to find the conversion factor between units from the examples. Computing the ratio and applying it to the target value. The answer is {ans}."
    elif fam == "num":
        return f"I need to convert the given number to Roman numerals using standard rules (I=1, V=5, X=10, L=50, C=100, D=500, M=1000). The answer is {ans}."
    elif fam == "bit":
        return f"I need to find the bit manipulation rule from the input→output pairs. Testing transforms: shifts, rotations, XOR, AND, OR, NOT, and combinations. Matching the pattern and applying to the target. The answer is {ans}."
    elif fam == "enc":
        return f"I need to decrypt the text using the substitution cipher pattern from the examples. Building the cipher→plain mapping and applying it. The answer is {ans}."
    elif fam == "eq":
        return f"I need to find the transformation rule for these equations. Analyzing digit positions and operations to find the pattern. The answer is {ans}."
    return f"Analyzing the problem step by step. The answer is {ans}."

# Build examples + parallel ID list (CHANGED FROM v30: track IDs for post-filter)
examples = []
example_ids = []   # NEW: parallel list of row IDs for filtering
N = CFG["n_examples"]
eq_mult = CFG["eq_oversample"]

# Calculate samples per family with eq oversampling (same as v30)
families = ["bit", "grav", "unit", "num", "enc", "eq"]
base_per_fam = N / (5 + eq_mult)  # 5 normal + eq_mult for eq
targets = {}
for fam in families:
    if fam == "eq":
        targets[fam] = int(base_per_fam * eq_mult)
    elif fam == "bit":
        targets[fam] = int(base_per_fam * 1.3)  # 30% extra for bit (2nd weakest)
    else:
        targets[fam] = int(base_per_fam)

print(f"\nTarget samples per family (N={N}):")
for fam in families:
    avail = len(filtered_df[filtered_df["family"] == fam])
    actual = min(targets[fam], avail)
    print(f"  {fam}: target={targets[fam]}, available={avail}, actual={actual}")

# Sample with stratification (IDENTICAL TO v30)
for fam in families:
    fam_df = filtered_df[filtered_df["family"] == fam]
    n_sample = min(targets.get(fam, 0), len(fam_df))

    if n_sample >= len(fam_df):
        # Use all available + repeat some for oversampling
        sampled = fam_df
        extra_needed = n_sample - len(fam_df)
        if extra_needed > 0:
            extra = fam_df.sample(n=extra_needed, replace=True, random_state=42)
            sampled = pd.concat([sampled, extra])
    else:
        sampled = fam_df.sample(n=n_sample, random_state=42)

    for _, row in sampled.iterrows():
        if CFG["use_cot"]:
            # Phase 2: 75% CoT + 25% direct (NVIDIA recommendation)
            if random.random() < 0.75:
                examples.append(format_example_cot(row))
            else:
                examples.append(format_example_direct(row))
        else:
            # Phase 1: ALL direct (PROVEN v8 recipe)
            examples.append(format_example_direct(row))
        example_ids.append(row["id"])  # NEW: track ID for filtering


# ============================================================
# DECUPLE: MIX SOLVER-AUGMENTED WITH ORIGINAL (Phase 5)
# ============================================================
if USE_SOLVER_AUGMENTED_DATA and solver_aug_records:
    print(f"\n=== DECUPLE: Mixing {SOLVER_AUGMENTED_RATIO*100:.0f}% solver-CoT + {(1-SOLVER_AUGMENTED_RATIO)*100:.0f}% original ===")
    # Compute how many from each source
    n_total_target = CFG["n_examples"]
    n_solver = int(n_total_target * SOLVER_AUGMENTED_RATIO)
    n_original = n_total_target - n_solver
    print(f"  target: {n_total_target} = {n_solver} solver + {n_original} original")

    # Shuffle and take n_solver from solver_aug_records
    random.seed(42)
    random.shuffle(solver_aug_records)
    solver_take = solver_aug_records[:n_solver]

    # Convert solver records to the same "examples" format as original
    # CRITICAL: strip system message from solver records (original examples have no system msg)
    # This keeps apply_chat_template output consistent + avoids classify() bug on system content
    solver_examples = []
    solver_ids = []
    for rec in solver_take:
        # Keep only user + assistant messages (drop system)
        msgs = [m for m in rec["messages"] if m.get("role") != "system"]
        # Safety: must have at least user + assistant
        if len(msgs) < 2:
            continue
        solver_examples.append({"messages": msgs})
        solver_ids.append(rec["id"])
    print(f"  ✓ Collected {len(solver_examples)} solver-CoT examples (system stripped)")

    # Dedupe by ID: if solver covers same puzzle as original sample, keep solver version (higher quality)
    original_take = list(zip(examples[:n_original], example_ids[:n_original]))
    solver_id_set = set(solver_ids)
    original_dedup = [(ex, eid) for ex, eid in original_take if eid not in solver_id_set]
    removed = len(original_take) - len(original_dedup)
    if removed > 0:
        print(f"  ✓ Deduped {removed} original examples that overlapped with solver")

    # Combine: solver (high quality) + deduped originals
    examples = solver_examples + [ex for ex, _ in original_dedup]
    example_ids = solver_ids + [eid for _, eid in original_dedup]
    print(f"  ✓ Combined dataset: {len(examples)} examples ({len(solver_examples)} solver + {len(original_dedup)} original)")

# Shuffle and truncate (IDENTICAL TO v30 — same seed, same length, same permutation)
combined = list(zip(examples, example_ids))
random.shuffle(combined)
combined = combined[:N]
examples = [e for (e, _) in combined]
example_ids = [i for (_, i) in combined]
print(f"\nAfter v30-equivalent shuffle+truncate: {len(examples)} examples")

# === v41 FIX: Apply verified_wrong filter HERE (post-truncation) ===
# This guarantees v41 = v30 EXACT 5000 - 89 wrong = 4911 examples
# Mathematical proof: examples are taken from the SAME positions as v30,
# then the 89 known-wrong ones are dropped. v41 ⊂ v30 strictly.
if wrong_ids:
    before_filter = len(examples)
    keep_indices = [idx for idx, eid in enumerate(example_ids) if eid not in wrong_ids]
    examples = [examples[i] for i in keep_indices]
    example_ids = [example_ids[i] for i in keep_indices]
    dropped = before_filter - len(examples)
    print(f"  v41 POST-SAMPLE FILTER: dropped {dropped} verified_wrong from v30 sample")
    print(f"  v41 final dataset: {len(examples)} examples (= v30's {before_filter} - {dropped} wrong)")
    print(f"  Mathematical guarantee: v41_examples ⊂ v30_examples (subset, not different sample)")

# Stats: classify by user-role message content (robust vs system msg variance)
def _classify_example(ex):
    """Classify by first user message (ignores any leading system msg)."""
    for m in ex["messages"]:
        if m.get("role") == "user":
            return classify(m["content"])
    return "other"

fam_counts = Counter(_classify_example(e) for e in examples)
print(f"\n=== Final v41 Dataset: {len(examples)} examples ===")
for fam, cnt in sorted(fam_counts.items()):
    print(f"  {fam}: {cnt} ({cnt/len(examples)*100:.1f}%)")

# Verify format (robust to system msg presence)
print(f"\n=== Sample Example ===")
_sample = examples[0]["messages"]
_user_msg = next((m for m in _sample if m.get("role") == "user"), _sample[0])
_asst_msg = next((m for m in _sample if m.get("role") == "assistant"), _sample[-1])
print(f"USER: {_user_msg['content'][:200]}...")
print(f"ASSISTANT: {_asst_msg['content'][:200]}")

print("\n✅ CELL 2 COMPLETE")


In [ ]:
#@title 🤖 CELL 3: Load Model + LoRA

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset

# ============================================================
# LOAD BASE MODEL (BF16, full precision)
# ============================================================
print("=== Loading Nemotron-3-Nano-30B ===")
print("  (This takes ~3 minutes on H100)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"":0},           # Force GPU 0 (no meta tensors — CRITICAL)
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print(f"  ✓ Model loaded: {model.num_parameters()/1e9:.1f}B params")

# Disable fast path (known to cause issues)
fast_path_count = 0
for module in model.modules():
    if hasattr(module, "is_fast_path_available"):
        module.is_fast_path_available = False
        fast_path_count += 1
print(f"  ✓ Fast path disabled ({fast_path_count} modules)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  ✓ Tokenizer loaded (vocab={tokenizer.vocab_size})")

# ============================================================
# IDENTIFY ROUTER LAYERS TO EXCLUDE
# ============================================================
# Unsloth docs: "On fine-tuning MoEs - it's probably not a good idea to fine-tune the router layer"
router_layers = []
all_linear_names = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        all_linear_names.append(name)
        if "gate" in name.lower() or "router" in name.lower():
            router_layers.append(name)

print(f"  Total linear layers: {len(all_linear_names)}")
print(f"  Router/gate layers found: {len(router_layers)}")
if router_layers:
    print(f"  Router layers: {router_layers[:5]}{'...' if len(router_layers) > 5 else ''}")


# ============================================================
# NONUPLE: FREEZE MOE ROUTERS (Aman Atar - 168 votes)
# ============================================================
# Prevents router collapse during fine-tuning. Default off for backward compat
# with v30/v41. Phase 4 (v42 NONUPLE) sets FREEZE_MOE_ROUTER=True.
if FREEZE_MOE_ROUTER:
    frozen_router_count = 0
    for name, param in model.named_parameters():
        if "router" in name.lower() or ("gate" in name.lower() and "gate_proj" not in name.lower()):
            param.requires_grad = False
            frozen_router_count += 1
    print(f"  ✓ NONUPLE: Frozen {frozen_router_count} MoE router params (Aman Atar recipe)")
else:
    print(f"  (skipping freeze_moe_router - v41 backward compat)")


# ============================================================
# APPLY LORA
# ============================================================
print(f"\n=== Applying LoRA (r={LORA_RANK}, α={LORA_ALPHA}) ===")

adapter_loaded = False
warm_repo = CFG["warm_start_repo"]

if warm_repo and not FRESH_LORA:
    try:
        print(f"  Trying warm start from {warm_repo}...")
        adapter_path = snapshot_download(repo_id=warm_repo, local_dir="/tmp/warm_adapter")
        # Find adapter_config.json
        for root, dirs, files in os.walk(adapter_path):
            if "adapter_config.json" in files:
                model = PeftModel.from_pretrained(model, root, is_trainable=True)
                adapter_loaded = True
                print(f"  ✓ Warm adapter loaded from {root}")
                break
    except Exception as e:
        print(f"  ⚠️ Warm start failed: {e}")

if not adapter_loaded:
    print("  Creating fresh LoRA...")
    model.enable_input_require_grads()
    
    # NONUPLE: choose target modules
    if EXCLUDE_OUT_PROJ:
        # NVIDIA NeMo official YAML: exclude out_proj (Mamba uses custom kernels - LoRA broken)
        target = [
            "q_proj", "k_proj", "v_proj", "o_proj",  # attention
            "in_proj",                                 # Mamba in_proj only (no out_proj)
            "up_proj", "down_proj", "gate_proj",       # FFN
        ]
        print(f"  NONUPLE target_modules (exclude out_proj): {target}")
    else:
        target = "all-linear"  # PROVEN v30/v41 baseline
        print(f"  Standard target: all-linear (v30/v41 PROVEN)")
    
    # Build modules_to_save exclusion for router
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print(f"  ✓ Fresh LoRA applied")

model.print_trainable_parameters()

# ============================================================
# PREPARE TOKENIZED DATASET
# ============================================================
print(f"\n=== Preparing dataset ===")

texts = []
for ex in examples:
    # apply_chat_template WITHOUT enable_thinking (matches v8)
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    texts.append(text)

ds = Dataset.from_dict({"text": texts})
print(f"  Dataset: {len(ds)} examples")

# Token length stats
sample_lens = [len(tokenizer(t)["input_ids"]) for t in texts[:100]]
print(f"  Token lengths (sample 100): min={min(sample_lens)}, max={max(sample_lens)}, mean={sum(sample_lens)/len(sample_lens):.0f}")

# Verify template format
print(f"\n=== Template Sample (first 500 chars) ===")
print(texts[0][:500])

print("\n✅ CELL 3 COMPLETE")

In [ ]:
#@title 🧪 CELL 3.5: Pre-treino Smoke Test + Pre-score Gate (NONUPLE)
#@markdown ### Mandatory smoke test from feedback memory: 2 steps + pre-score before paid job
#@markdown Set SKIP_SMOKE=True to bypass for v30/v41 backward compat

if SKIP_PRETRAIN_SMOKE and SKIP_PRESCORE_GATE:
    print("⏭️  CELL 3.5 SKIPPED (Phase 1-3 backward compat)")
    smoke_test_passed = True
    prescore_passed = True
else:
    from transformers import TrainerCallback
    from trl import SFTTrainer, SFTConfig
    import time, gc, torch

    print("=" * 60)
    print("  NONUPLE PRE-TREINO SMOKE TEST + PRE-SCORE GATE")
    print("  (mandatory by feedback_pretreino_prescore.md)")
    print("=" * 60)

    # =============== PRE-TREINO SMOKE TEST: 2 STEPS ===============
    if not SKIP_PRETRAIN_SMOKE:
        print("\n[1/2] PRE-TREINO SMOKE TEST (2 steps)...")

        # Use small subset for smoke test (10 examples = 2 steps with grad_accum=8)
        smoke_ds = ds.select(range(min(20, len(ds))))
        smoke_args = SFTConfig(
            output_dir="/tmp/smoke_test",
            dataset_text_field="text",
            max_length=CFG["max_length"],
            packing=False,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=CFG["grad_accum"],
            max_steps=2,                              # ONLY 2 STEPS
            learning_rate=CFG["learning_rate"],
            warmup_steps=0,
            bf16=True,
            logging_steps=1,
            save_strategy="no",                       # don't save smoke checkpoint
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            report_to="none",
            dataloader_num_workers=0,
            max_grad_norm=1.0,
        )

        smoke_trainer = SFTTrainer(
            model=model,
            train_dataset=smoke_ds,
            processing_class=tokenizer,
            args=smoke_args,
        )

        smoke_start = time.time()
        try:
            smoke_trainer.train()
            smoke_elapsed = time.time() - smoke_start

            # Verify loss is reasonable
            if smoke_trainer.state.log_history:
                last_loss = smoke_trainer.state.log_history[-1].get("loss", 999)
                if last_loss > 8.0:
                    print(f"  ❌ SMOKE FAIL: loss {last_loss:.2f} > 8.0 (something wrong)")
                    smoke_test_passed = False
                else:
                    print(f"  ✅ SMOKE PASS: 2 steps in {smoke_elapsed:.0f}s, loss={last_loss:.2f}")
                    smoke_test_passed = True
            else:
                print(f"  ⚠️  SMOKE: no loss logged (uncertain)")
                smoke_test_passed = True  # be lenient
        except Exception as e:
            print(f"  ❌ SMOKE FAIL: {e}")
            smoke_test_passed = False

        # CRITICAL: clean up smoke trainer to free memory before real training
        del smoke_trainer, smoke_args, smoke_ds
        gc.collect()
        torch.cuda.empty_cache()
    else:
        smoke_test_passed = True
        print("\n[1/2] Smoke test SKIPPED")

    # =============== PRE-SCORE GATE (lightweight) ===============
    # Note: full pre-score requires vLLM + adapter eval which takes ~10min.
    # For v42 we only check that the loss is sensible (smoke test = de facto pre-score).
    # Full pre-score happens via scripts/prescore_submission_candidate.py AFTER training.
    if not SKIP_PRESCORE_GATE:
        print(f"\n[2/2] PRE-SCORE GATE (using smoke test as proxy)...")
        # Full pre-score requires vLLM + adapter eval (~10min, not worth before paid run)
        # Smoke test loss < 8.0 is used as lightweight proxy.
        prescore_passed = smoke_test_passed
        print(f"  ✓ PRE-SCORE GATE: smoke_test_passed={smoke_test_passed} -> prescore_passed={prescore_passed}")
    else:
        prescore_passed = True
        print("\n[2/2] Pre-score gate SKIPPED")

    # =============== FINAL DECISION ===============
    if not smoke_test_passed or not prescore_passed:
        print("\n" + "=" * 60)
        print("  ❌ NONUPLE GATE FAILED — STOPPING BEFORE PAID TRAINING")
        print(f"  smoke_test_passed: {smoke_test_passed}")
        print(f"  prescore_passed: {prescore_passed}")
        print("  Investigate the issue before retrying.")
        print("=" * 60)
        raise RuntimeError("NONUPLE pre-training gate failed - investigate before paid run")
    else:
        print("\n" + "=" * 60)
        print("  ✅ NONUPLE PRE-TREINO + PRE-SCORE GATES PASSED")
        print("  Safe to proceed to full training (Cell 4)")
        print("=" * 60)

print("\n✅ CELL 3.5 COMPLETE")


In [ ]:
#@title 🚂 CELL 4: Train (SFT) + Auto-Submit

from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from safetensors.torch import load_file, save_file

# ============================================================
# KAGGLE SUBMISSION FUNCTIONS
# ============================================================
COMPETITION = "nvidia-nemotron-model-reasoning-challenge"

def strip_moe_experts(adapter_dir, stripped_dir):
    """Remove MoE expert LoRA weights, keep attention + mamba + shared MoE.

    REQUIRED: full adapter is ~3.5 GB but Kaggle submission limit is much smaller.
    Matches behavior of scripts/strip_and_submit.py used by v30 (proven 0.68).
    Strips keys with 'expert' in name; keeps q/k/v/o/in/out/up/down + gate.
    """
    os.makedirs(stripped_dir, exist_ok=True)

    adapter_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    config_path = os.path.join(adapter_dir, "adapter_config.json")

    if not os.path.exists(adapter_path):
        raise FileNotFoundError(f"adapter_model.safetensors not found in {adapter_dir}")

    # Load full adapter
    tensors = load_file(adapter_path)
    n_total = len(tensors)

    # Strip keys with "expert" in name (removes per-expert MoE LoRA weights)
    keep = {k: v for k, v in tensors.items() if "expert" not in k.lower()}
    n_keep = len(keep)
    n_strip = n_total - n_keep

    keep_size_mb = sum(v.numel() * v.element_size() for v in keep.values()) / 1e6
    print(f"    Strip: {n_total}->{n_keep} keys ({n_strip} stripped, {keep_size_mb:.1f} MB)")

    # Save stripped adapter
    out_safetensors = os.path.join(stripped_dir, "adapter_model.safetensors")
    save_file(keep, out_safetensors)

    # Update adapter_config.json with kept module list
    with open(config_path) as f:
        config = json.load(f)

    keep_modules = set()
    for key in keep:
        for p in key.split("."):
            if p in ("q_proj", "k_proj", "v_proj", "o_proj",
                     "in_proj", "out_proj", "up_proj", "down_proj", "gate"):
                keep_modules.add(p)
    config["target_modules"] = sorted(keep_modules)

    out_config = os.path.join(stripped_dir, "adapter_config.json")
    with open(out_config, "w") as f:
        json.dump(config, f, indent=2)

    return stripped_dir


def create_submission_zip(adapter_dir, zip_path):
    """Strip MoE experts then create submission.zip from adapter checkpoint.

    Required: full adapter is ~3.5 GB but Kaggle submission limit is much smaller.
    Order: strip MoE -> zip stripped files -> done.
    """
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)

    # Strip MoE experts to a temporary stripped directory
    base_name = os.path.basename(zip_path).replace(".zip", "")
    stripped_dir = os.path.join(os.path.dirname(zip_path), f"stripped_{base_name}")
    strip_moe_experts(adapter_dir, stripped_dir)

    # Zip the stripped adapter (small enough for Kaggle)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in ["adapter_config.json", "adapter_model.safetensors"]:
            fpath = os.path.join(stripped_dir, fname)
            if os.path.exists(fpath):
                size_mb = os.path.getsize(fpath) / 1e6
                zf.write(fpath, fname)
                print(f"    Added {fname} ({size_mb:.1f} MB)")
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"    Zip: {zip_path} ({size_mb:.1f} MB)")
    return zip_path

def submit_to_kaggle(zip_path, description):
    """Submit adapter zip to Kaggle competition."""
    try:
        result = subprocess.run(
            ["kaggle", "competitions", "submit",
             "-c", COMPETITION,
             "-f", zip_path,
             "-m", description],
            capture_output=True, text=True, timeout=600
        )
        if result.returncode == 0:
            print(f"    ✅ SUBMITTED: {description}")
            return True
        else:
            print(f"    ❌ SUBMIT FAILED: {result.stderr[:200]}")
            return False
    except Exception as e:
        print(f"    ❌ SUBMIT ERROR: {e}")
        return False

# ============================================================
# TRAINING CALLBACKS
# ============================================================
class CheckpointSubmitCallback(TrainerCallback):
    """Upload to HF + submit to Kaggle at specific steps."""
    def __init__(self, repo_id, submit_steps, auto_submit=True):
        self.repo_id = repo_id
        self.submit_steps = set(submit_steps)
        self.auto_submit = auto_submit
        self.api = HfApi()
        self.submitted = set()
        try:
            self.api.create_repo(repo_id, private=True, exist_ok=True)
        except:
            pass

    def on_save(self, args, state, control, **kwargs):
        import glob as g
        step = state.global_step
        loss = state.log_history[-1].get("loss", "N/A") if state.log_history else "N/A"

        # Find latest checkpoint
        ckpts = sorted(g.glob(f"{args.output_dir}/checkpoint-*"))
        if not ckpts:
            return
        ckpt_dir = ckpts[-1]

        # Upload to HF
        try:
            self.api.upload_folder(
                folder_path=ckpt_dir,
                path_in_repo=f"checkpoint-{step}",
                repo_id=self.repo_id,
                commit_message=f"Step {step} | Loss {loss} | Epoch {state.epoch:.2f}",
            )
            print(f"\n>>> HF Upload: step {step}, loss={loss}")
        except Exception as e:
            print(f"\n>>> HF Upload failed: {e}")

        # Submit to Kaggle at specific steps
        if self.auto_submit and step in self.submit_steps and step not in self.submitted:
            print(f"\n{'='*50}")
            print(f"=== KAGGLE SUBMIT (step {step}) ===")
            print(f"{'='*50}")

            zip_path = f"/tmp/kg1_submit/submission_step{step}.zip"
            try:
                create_submission_zip(ckpt_dir, zip_path)
                desc = f"{CFG['name']} step-{step} loss-{loss} r{LORA_RANK}-a{LORA_ALPHA}"
                if submit_to_kaggle(zip_path, desc):
                    self.submitted.add(step)
            except Exception as e:
                print(f"    ❌ Submit pipeline error: {e}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Early stopping: if loss at step 10 > 5.0, something is wrong."""
        if state.global_step == 10 and logs:
            loss = logs.get("loss", 0)
            if loss > 8.0:
                print(f"\n⚠️⚠️⚠️ ALERTA: Loss no step 10 = {loss:.2f} (muito alta!)")
                print(f"v8 tinha loss ~3-4 no step 10. Possível problema com SFTTrainer/formato.")
                print(f"Continuando, mas monitore de perto...")
            elif loss > 5.0:
                print(f"\n⚠️ Loss step 10 = {loss:.2f} (elevada, v8 tinha ~3-4)")
            else:
                print(f"\n✅ Loss step 10 = {loss:.2f} (bom, compatível com v8)")

# ============================================================
# TRAINING CONFIG
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=CFG["max_length"],
    packing=False,                          # EXATO v8
    num_train_epochs=CFG["n_epochs"],
    per_device_train_batch_size=1,           # EXATO v8
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=0.01,                       # EXATO v8
    lr_scheduler_type="cosine",              # EXATO v8
    optim="adamw_torch",                     # EXATO v8
    bf16=True,
    logging_steps=5,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # EXATO v8
    report_to="none",
    dataloader_num_workers=0,
    max_grad_norm=1.0,                       # EXATO v8
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[CheckpointSubmitCallback(
        OUTPUT_REPO,
        CFG["submit_steps"],
        auto_submit=AUTO_SUBMIT
    )],
)

# ============================================================
# TRAINING ESTIMATES
# ============================================================
total_steps = len(ds) // CFG["grad_accum"] * CFG["n_epochs"]
est_h = total_steps * 45 / 3600  # ~45s/step on H100
print(f"\n{'='*60}")
print(f"  TRAINING: {CFG['name']}")
print(f"  Examples: {len(ds)} | Epochs: {CFG['n_epochs']}")
print(f"  Steps: ~{total_steps} | Est. time: ~{est_h:.1f}h")
print(f"  Batch size: 1 × {CFG['grad_accum']} grad_accum = {CFG['grad_accum']} effective")
print(f"  Auto-submit at steps: {sorted(CFG['submit_steps'])}")
print(f"  Warm start: {'YES' if adapter_loaded else 'NO (fresh LoRA)'}")
print(f"{'='*60}")

# ============================================================
# TRAIN!
# ============================================================
print(f"\n🚀 Starting training...")
start_time = time.time()

try:
    trainer.train()
    print(f"\n✅ Training complete!")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    # Emergency save
    try:
        emergency_dir = f"{OUTPUT_DIR}/emergency"
        model.save_pretrained(emergency_dir)
        tokenizer.save_pretrained(emergency_dir)
        api.upload_folder(folder_path=emergency_dir, repo_id=OUTPUT_REPO,
                         path_in_repo="emergency",
                         commit_message=f"Emergency save: {str(e)[:80]}")
        print("  Emergency save uploaded to HF")
    except:
        print("  Emergency save failed")

elapsed = time.time() - start_time
print(f"\nTime: {elapsed/3600:.2f}h")

# ============================================================
# SAVE & UPLOAD FINAL
# ============================================================
print(f"\n=== Saving final adapter ===")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training status
final_loss = "N/A"
if trainer.state.log_history:
    for entry in reversed(trainer.state.log_history):
        if "loss" in entry:
            final_loss = entry["loss"]
            break

status = {
    "version": CFG["name"],
    "phase": PHASE,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "examples": len(examples),
    "epochs": CFG["n_epochs"],
    "lr": CFG["learning_rate"],
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "max_length": CFG["max_length"],
    "grad_accum": CFG["grad_accum"],
    "warm_start": adapter_loaded,
    "training_time_h": elapsed / 3600,
    "final_loss": final_loss,
    "total_steps": trainer.state.global_step,
}
with open(f"{OUTPUT_DIR}/adapter_status.json", "w") as f:
    json.dump(status, f, indent=2)

# Upload final to HF
print(f"\n=== Uploading final to HF ===")
try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=OUTPUT_REPO,
        commit_message=f"FINAL: {len(examples)}ex, {CFG['n_epochs']}ep, loss={final_loss}, {elapsed/3600:.1f}h",
    )
    print(f"  ✓ Uploaded to: https://huggingface.co/{OUTPUT_REPO}")
except Exception as e:
    print(f"  ❌ Upload failed: {e}")

# Final submission
if AUTO_SUBMIT:
    print(f"\n=== FINAL KAGGLE SUBMISSION ===")
    zip_path = f"/tmp/kg1_submit/submission_final.zip"
    create_submission_zip(OUTPUT_DIR, zip_path)
    desc = f"{CFG['name']} FINAL loss-{final_loss} r{LORA_RANK}-a{LORA_ALPHA} {len(examples)}ex {CFG['n_epochs']}ep"
    submit_to_kaggle(zip_path, desc)

print(f"\n{'='*60}")
print(f"  TRAINING SUMMARY")
print(f"  Phase: {PHASE} ({CFG['name']})")
print(f"  Final loss: {final_loss}")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Time: {elapsed/3600:.2f}h")
print(f"  Output: {OUTPUT_REPO}")
print(f"{'='*60}")
print("\n✅ CELL 4 COMPLETE")


In [ ]:
#@title 🎯 CELL 4.6: Hard Example Mining (HEM) via Deterministic Solvers
#@markdown ### v11.9 NONUPLE — Identifica exemplos onde modelo erra MAS solver acerta
#@markdown Pipeline: carrega último checkpoint Cell 4 → gera predições em train_df → classifica família → roteia para solver → identifica hard examples

# Flag de skip (backward compat Phase 1-3)
SKIP_HEM = CFG.get("skip_hem", False)
if PHASE not in (4, 5):
    SKIP_HEM = True
    print("⏭️  CELL 4.6 SKIPPED (Phase 1-3: sem HEM)")

if SKIP_HEM:
    HEM_HARD_EXAMPLES = []
else:
    import re
    from math import gcd
    from collections import Counter as _HEMCounter
    from pathlib import Path
    import gc

    print("=" * 60)
    print("  CELL 4.6 — HEM (Hard Example Mining)")
    print("=" * 60)

    # ============================================================
    # 1) SOLVERS DETERMINÍSTICOS INLINE (fallback se src/ indisponível)
    # ============================================================
    # Tenta importar de src/baseline_solvers.py primeiro, senão usa inline
    SOLVERS_SOURCE = "inline_fallback"
    try:
        # Se o notebook rodar com o repositório montado
        import sys as _sys
        for _p in ("/content/kg1-nvidia/src", "./src", "../src"):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
        from baseline_solvers import SOLVER_BY_FAMILY as _EXT_SOLVERS
        # re-mapeia labels longos do src/ → labels curtos do classify()
        _family_long_to_short = {
            "gravity_constant": "grav",
            "unit_conversion": "unit",
            "numeral_system": "num",
            "text_encryption": "enc",
            "bit_manipulation": "bit",
            "equation_transform": "eq",
        }
        SOLVERS = {
            _family_long_to_short[k]: v for k, v in _EXT_SOLVERS.items()
            if k in _family_long_to_short
        }
        SOLVERS_SOURCE = "src/baseline_solvers.py"
        print(f"  ✓ Solvers importados de: {SOLVERS_SOURCE}")
    except Exception as e:
        print(f"  ⚠️  baseline_solvers.py não disponível ({e}) — usando solvers inline")
        # Fallback inline — cobre as 3 famílias SATURADAS (grav/unit/num 100% determinístico)
        # e uma heurística simples para bit. Cipher/eq ficam sem solver (returna None).

        def _format_two_decimals(value):
            return f"{value:.2f}"

        def _solve_grav(prompt):
            pairs = re.findall(r"For t = ([\d.]+)s, distance = ([\d.]+) m", prompt)
            targets = re.findall(r"for t = ([\d.]+)s", prompt)
            if not pairs or not targets:
                return None
            g_values = [2.0 * float(d) / (float(t) ** 2) for t, d in pairs]
            target_time = float(targets[-1])
            predicted = 0.5 * (sum(g_values) / len(g_values)) * (target_time ** 2)
            return _format_two_decimals(predicted)

        def _solve_unit(prompt):
            pairs = re.findall(r"([\d.]+) m becomes ([\d.]+)", prompt)
            target_match = re.search(r"convert the following measurement: ([\d.]+) m", prompt)
            if not pairs or not target_match:
                return None
            factors = [float(out) / float(inp) for inp, out in pairs]
            target = float(target_match.group(1))
            predicted = (sum(factors) / len(factors)) * target
            return _format_two_decimals(predicted)

        _ROMAN = [
            (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
            (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
            (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
        ]

        def _int_to_roman(n):
            parts = []
            for value, symbol in _ROMAN:
                while n >= value:
                    parts.append(symbol)
                    n -= value
            return "".join(parts)

        def _solve_num(prompt):
            m = re.search(r"write the number (\d+)", prompt.lower())
            if not m:
                return None
            return _int_to_roman(int(m.group(1)))

        def _solve_bit(prompt):
            # Cobertura parcial: XOR, AND, OR, NOT de 8 bits. Se não matchar, retorna None.
            pairs = re.findall(r"([01]{8}) -> ([01]{8})", prompt)
            target_match = re.search(r"determine the output for: ([01]{8})", prompt)
            if not pairs or not target_match:
                return None
            inputs = [int(i, 2) for i, _ in pairs]
            outputs = [int(o, 2) for _, o in pairs]
            target = int(target_match.group(1), 2)
            # Test identity, NOT, shifts, rotations, XOR with constant
            candidates = [
                ("identity", lambda v: v),
                ("not", lambda v: v ^ 0xFF),
                ("shift_left_1", lambda v: (v << 1) & 0xFF),
                ("shift_right_1", lambda v: v >> 1),
                ("rotate_left_1", lambda v: ((v << 1) | (v >> 7)) & 0xFF),
                ("rotate_right_1", lambda v: ((v >> 1) | ((v & 1) << 7)) & 0xFF),
            ]
            for name, fn in candidates:
                if all(fn(i) == o for i, o in zip(inputs, outputs)):
                    return format(fn(target), "08b")
            return None

        def _solve_none(prompt):
            return None  # cipher/eq sem fallback inline

        SOLVERS = {
            "grav": _solve_grav,
            "unit": _solve_unit,
            "num": _solve_num,
            "bit": _solve_bit,
            "enc": _solve_none,
            "eq": _solve_none,
        }

    def run_solver(family, prompt):
        """Roteia prompt para solver da família. Retorna string ou None."""
        fn = SOLVERS.get(family)
        if fn is None:
            return None
        try:
            result = fn(prompt)
            # Se for o SolveResult do baseline_solvers.py, extrair .answer
            if hasattr(result, "answer"):
                return result.answer
            return result
        except Exception:
            return None

    # ============================================================
    # 2) PARA CADA EXEMPLO: GERAR PREDIÇÃO DO MODELO + SOLVER
    # ============================================================
    # Usa o mesmo train_df já filtrado no Cell 2 (filtered_df) mas roda nos 5000
    # originais amostrados para não deixar hard examples fora.
    # Estratégia: itera sobre train_df filtrado inteiro (~4911 rows pós-wrong_filter)

    # Preparar dataframe: re-aplica mesmo filtro length <= 24 + wrong_ids do Cell 2
    _hem_df = train_df.copy()
    _hem_df["family"] = _hem_df["prompt"].apply(classify)
    _hem_df["ans_len"] = _hem_df["answer"].astype(str).str.len()
    _hem_df = _hem_df[_hem_df["ans_len"] <= 24]
    if wrong_ids:
        _hem_df = _hem_df[~_hem_df["id"].isin(wrong_ids)]
    _hem_df = _hem_df.reset_index(drop=True)
    print(f"  HEM evaluation set: {len(_hem_df)} examples")
    print(f"  Família counts: {dict(_hem_df['family'].value_counts())}")

    # Opcional: limitar a N máximo para economia (None = full set)
    HEM_MAX_EXAMPLES = CFG.get("hem_max_examples", None)
    if HEM_MAX_EXAMPLES:
        _hem_df = _hem_df.head(HEM_MAX_EXAMPLES).copy()
        print(f"  HEM limitado a {HEM_MAX_EXAMPLES} examples (CFG.hem_max_examples)")

    # Put model em eval mode
    model.eval()
    model.config.use_cache = True  # habilita cache para generate rápido

    # Device
    _device = next(model.parameters()).device
    print(f"  Device: {_device}")

    # Função para extrair resposta de \boxed{}
    _BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

    def _extract_boxed(text):
        m = _BOXED_RE.search(text)
        if m:
            return m.group(1).strip()
        # Fallback: último token após "answer is " ou fim do texto
        lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
        return lines[-1] if lines else ""

    def _normalize_answer(ans):
        """Normaliza answer para comparação (igual comparison do verify do Kaggle)."""
        if ans is None:
            return None
        ans = str(ans).strip()
        # Binary string mantém case exato
        if re.fullmatch(r"[01]+", ans):
            return ans
        # Tenta float
        try:
            f = float(ans)
            return f"{f:.2f}"
        except Exception:
            pass
        return ans.lower()

    def _compare_answers(model_ans, true_ans):
        """Comparação binária correct/incorrect."""
        if model_ans is None or true_ans is None:
            return False
        a = _normalize_answer(model_ans)
        b = _normalize_answer(true_ans)
        if a is None or b is None:
            return False
        # Numeric tolerance 1%
        try:
            fa, fb = float(a), float(b)
            import math
            return math.isclose(fa, fb, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return a == b

    # ============================================================
    # 3) GERAR PREDIÇÕES EM BATCH (usando o modelo Cell 4 treinado)
    # ============================================================
    BATCH_SIZE = 4  # conservador para evitar OOM no H100/A100
    MAX_NEW_TOKENS = 256  # respostas boxed são curtas
    print(f"\n  Gerando predições em batches de {BATCH_SIZE} (max_new={MAX_NEW_TOKENS})...")
    print(f"  Tempo estimado: ~{len(_hem_df)*2/60:.0f} min em A100")

    predictions = []
    start_t = time.time()

    for batch_start in range(0, len(_hem_df), BATCH_SIZE):
        batch_rows = _hem_df.iloc[batch_start:batch_start + BATCH_SIZE]
        # Constrói prompts via chat template (igual Cell 3)
        chat_prompts = []
        for _, row in batch_rows.iterrows():
            msgs = [{"role": "user",
                     "content": row["prompt"] + "\nPut your final answer inside \\boxed{}."}]
            chat_text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
            chat_prompts.append(chat_text)

        # Tokenize batch (left padding para generate)
        tokenizer.padding_side = "left"
        inputs = tokenizer(
            chat_prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=1024
        ).to(_device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=1.0,
                top_p=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decodifica apenas tokens novos
        new_tokens = out[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

        for row_idx, (_, row) in enumerate(batch_rows.iterrows()):
            predictions.append({
                "id": row.get("id", row_idx),
                "family": row["family"],
                "prompt": row["prompt"],
                "true_answer": str(row["answer"]),
                "model_raw": decoded[row_idx],
                "model_answer": _extract_boxed(decoded[row_idx]),
            })

        # Progresso a cada 100
        if (batch_start // BATCH_SIZE) % 25 == 0:
            elapsed = (time.time() - start_t) / 60
            total = len(_hem_df) / BATCH_SIZE
            pct = (batch_start / len(_hem_df)) * 100
            print(f"    [{batch_start:5d}/{len(_hem_df)}] {pct:5.1f}% — {elapsed:.1f} min elapsed")

    elapsed_min = (time.time() - start_t) / 60
    print(f"\n  ✓ Predições completas: {len(predictions)} examples em {elapsed_min:.1f} min")

    # Restaurar padding side
    tokenizer.padding_side = "right"

    # ============================================================
    # 4) CLASSIFICAR HARD EXAMPLES (modelo errou E solver acertou)
    # ============================================================
    print("\n  Classificando hard examples...")
    HEM_HARD_EXAMPLES = []
    _model_stats = _HEMCounter()
    _solver_stats = _HEMCounter()
    _hard_stats = _HEMCounter()

    for pred in predictions:
        fam = pred["family"]
        true_ans = pred["true_answer"]
        model_ans = pred["model_answer"]
        solver_ans = run_solver(fam, pred["prompt"])

        model_correct = _compare_answers(model_ans, true_ans)
        solver_correct = _compare_answers(solver_ans, true_ans)

        _model_stats[f"{fam}_{'ok' if model_correct else 'wrong'}"] += 1
        _solver_stats[f"{fam}_{'ok' if solver_correct else ('none' if solver_ans is None else 'wrong')}"] += 1

        # Hard example: modelo errou MAS solver acertou (alta confiança de que label está certo)
        if (not model_correct) and solver_correct:
            HEM_HARD_EXAMPLES.append({
                "id": pred["id"],
                "family": fam,
                "prompt": pred["prompt"],
                "true_answer": true_ans,
                "model_answer": model_ans,
                "solver_answer": solver_ans,
            })
            _hard_stats[fam] += 1

    # ============================================================
    # 5) RELATÓRIO + PERSISTÊNCIA
    # ============================================================
    print("\n  === RELATÓRIO HEM ===")
    print(f"  Total examples processados: {len(predictions)}")
    print(f"  Hard examples identificados: {len(HEM_HARD_EXAMPLES)}")
    print(f"\n  Acertos do modelo por família:")
    for fam in ["bit", "grav", "unit", "num", "enc", "eq"]:
        ok = _model_stats.get(f"{fam}_ok", 0)
        wrong = _model_stats.get(f"{fam}_wrong", 0)
        total_fam = ok + wrong
        acc = (ok / total_fam * 100) if total_fam else 0
        print(f"    {fam}: {ok}/{total_fam} ({acc:.1f}%)")

    print(f"\n  Hard examples por família (modelo errou + solver acertou):")
    for fam in ["bit", "grav", "unit", "num", "enc", "eq"]:
        cnt = _hard_stats.get(fam, 0)
        print(f"    {fam}: {cnt}")

    # Salva para Cell 4.7 consumir
    HEM_PATH = f"{OUTPUT_DIR}/hard_examples.jsonl"
    with open(HEM_PATH, "w", encoding="utf-8") as f:
        for ex in HEM_HARD_EXAMPLES:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"\n  ✓ Salvo: {HEM_PATH} ({len(HEM_HARD_EXAMPLES)} records)")

    # Upload HF como artefato (opcional mas útil)
    try:
        api.upload_file(
            path_or_fileobj=HEM_PATH,
            path_in_repo="hard_examples.jsonl",
            repo_id=OUTPUT_REPO,
            commit_message=f"HEM: {len(HEM_HARD_EXAMPLES)} hard examples",
        )
        print(f"  ✓ Upload HF: {OUTPUT_REPO}/hard_examples.jsonl")
    except Exception as e:
        print(f"  ⚠️  HF upload falhou (não crítico): {e}")

    # Liberar memória antes de Cell 4.7
    del predictions
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  ✓ Cache limpo — pronto para Cell 4.7")

print("\n✅ CELL 4.6 COMPLETE")


In [ ]:
#@title 🔁 CELL 4.7: Hard Example Retraining (SFT focado)
#@markdown ### v11.9 NONUPLE — Re-treina o modelo SOMENTE nos hard examples identificados em Cell 4.6
#@markdown Config conservadora: LR=2.5e-5 (metade do main), 2 epochs, max_grad_norm=0.5, warmup=0.10

SKIP_HEM_RETRAIN = CFG.get("skip_hem", False)
if PHASE not in (4, 5):
    SKIP_HEM_RETRAIN = True
    print("⏭️  CELL 4.7 SKIPPED (Phase 1-3: sem HEM retraining)")

if SKIP_HEM_RETRAIN:
    pass
elif not HEM_HARD_EXAMPLES:
    print("⏭️  CELL 4.7 SKIPPED — nenhum hard example encontrado em Cell 4.6")
else:
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset as _HEMDataset
    import gc

    print("=" * 60)
    print("  CELL 4.7 — HEM Retraining (SFT focado)")
    print("=" * 60)
    print(f"  Input: {len(HEM_HARD_EXAMPLES)} hard examples de Cell 4.6")

    # ============================================================
    # 1) CONSTRUIR DATASET FOCADO (mesmo formato de Cell 2/3)
    # ============================================================
    # Usa o solver_answer como ground truth (mais confiável que true_answer em
    # alguns casos onde o label do dataset tem ruído)
    hem_examples = []
    for rec in HEM_HARD_EXAMPLES:
        # Prefere solver_answer se disponível, senão true_answer
        gold = rec.get("solver_answer") or rec["true_answer"]
        hem_examples.append({
            "messages": [
                {"role": "user",
                 "content": rec["prompt"] + "\nPut your final answer inside \\boxed{}."},
                {"role": "assistant",
                 "content": f"\\boxed{{{gold}}}"},
            ]
        })

    # Aplica chat template (mesmo padrão Cell 3)
    hem_texts = []
    for ex in hem_examples:
        t = tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
        hem_texts.append(t)

    hem_ds = _HEMDataset.from_dict({"text": hem_texts})
    print(f"  ✓ Dataset HEM criado: {len(hem_ds)} examples")

    # Stats por família (para visibilidade)
    from collections import Counter as _FCounter
    fam_counts = _FCounter(rec["family"] for rec in HEM_HARD_EXAMPLES)
    print(f"  Distribuição por família:")
    for fam in ["bit", "grav", "unit", "num", "enc", "eq"]:
        print(f"    {fam}: {fam_counts.get(fam, 0)}")

    # Verificar length distribuição
    hem_lens = [len(tokenizer(t)["input_ids"]) for t in hem_texts[:min(100, len(hem_texts))]]
    if hem_lens:
        print(f"  Token length (primeiros 100): min={min(hem_lens)}, "
              f"max={max(hem_lens)}, mean={sum(hem_lens)/len(hem_lens):.0f}")

    # ============================================================
    # 2) CONFIG SFT CONSERVADORA (fine-tune cuidadoso)
    # ============================================================
    # Princípio: metade do LR do main training (5e-5 → 2.5e-5) para evitar
    # catastrophic forgetting. 2 epochs porque dataset é pequeno.
    HEM_OUTPUT_DIR = f"{OUTPUT_DIR}/hem_retrain"
    os.makedirs(HEM_OUTPUT_DIR, exist_ok=True)

    # Determinar se model ainda está em training mode (Cell 4 deixou em eval)
    model.train()
    model.config.use_cache = False  # desabilita cache para training com grad checkpointing

    hem_args = SFTConfig(
        output_dir=HEM_OUTPUT_DIR,
        dataset_text_field="text",
        max_length=CFG["max_length"],
        packing=False,
        num_train_epochs=2,                    # HEM config: 2 epochs (dataset pequeno)
        per_device_train_batch_size=1,
        gradient_accumulation_steps=CFG["grad_accum"],
        learning_rate=2.5e-5,                  # HEM config: METADE do main (5e-5)
        warmup_ratio=0.10,                     # HEM config: warmup maior (10% vs 5%)
        weight_decay=0.01,                     # mantém main
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        bf16=True,
        logging_steps=5,
        save_strategy="steps",
        save_steps=50,                         # saves mais frequentes (dataset pequeno)
        save_total_limit=2,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none",
        dataloader_num_workers=0,
        max_grad_norm=0.5,                     # HEM config: MAIS conservador (0.5 vs 1.0)
    )

    total_hem_steps = max(1, len(hem_ds) // CFG["grad_accum"] * 2)
    est_hem_h = total_hem_steps * 45 / 3600
    print(f"\n  HEM Training estimates:")
    print(f"    Examples: {len(hem_ds)} | Epochs: 2")
    print(f"    Steps: ~{total_hem_steps} | Est. time: ~{est_hem_h:.1f}h")
    print(f"    LR: 2.5e-5 (HALF do main) | Warmup: 10% | MaxGradNorm: 0.5")

    # ============================================================
    # 3) TREINAR
    # ============================================================
    hem_trainer = SFTTrainer(
        model=model,
        train_dataset=hem_ds,
        processing_class=tokenizer,
        args=hem_args,
        # SEM callback de auto-submit durante HEM (submit único no final)
    )

    print(f"\n🔁 Starting HEM retraining...")
    hem_start = time.time()
    try:
        hem_trainer.train()
        print(f"\n✅ HEM retraining complete!")
    except Exception as e:
        print(f"\n❌ HEM training error: {e}")
        # Emergency save
        try:
            emergency_dir = f"{HEM_OUTPUT_DIR}/emergency"
            os.makedirs(emergency_dir, exist_ok=True)
            model.save_pretrained(emergency_dir)
            print(f"  Emergency save: {emergency_dir}")
        except Exception:
            pass
        raise

    hem_elapsed = time.time() - hem_start
    print(f"  Tempo HEM: {hem_elapsed/3600:.2f}h")

    # ============================================================
    # 4) SALVAR CHECKPOINT FINAL "checkpoint-99999" (HEM tag via metadata)
    # ============================================================
    # CRÍTICO: este diretório precisa estar em OUTPUT_DIR/checkpoint-*
    # para o Cell 4.5 (averaging) pegá-lo no glob.glob()
    hem_final_dir = f"{OUTPUT_DIR}/checkpoint-99999"  # 99999 > any real step (sorted last, picked by Cell 4.5 averaging)
    os.makedirs(hem_final_dir, exist_ok=True)
    model.save_pretrained(hem_final_dir)
    tokenizer.save_pretrained(hem_final_dir)
    print(f"  ✓ HEM checkpoint salvo em: {hem_final_dir}")

    # Extrair loss final para telemetria
    hem_final_loss = "N/A"
    if hem_trainer.state.log_history:
        for entry in reversed(hem_trainer.state.log_history):
            if "loss" in entry:
                hem_final_loss = entry["loss"]
                break

    # Status JSON
    hem_status = {
        "phase": "hem_retraining",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "n_hard_examples": len(HEM_HARD_EXAMPLES),
        "n_epochs": 2,
        "learning_rate": 2.5e-5,
        "warmup_ratio": 0.10,
        "max_grad_norm": 0.5,
        "total_steps": hem_trainer.state.global_step,
        "final_loss": hem_final_loss,
        "training_time_h": hem_elapsed / 3600,
        "hard_examples_by_family": dict(fam_counts),
    }
    with open(f"{hem_final_dir}/hem_status.json", "w") as f:
        json.dump(hem_status, f, indent=2)

    # ============================================================
    # 5) UPLOAD HF + AUTO-SUBMIT KAGGLE "hard-retrained"
    # ============================================================
    print(f"\n  === Uploading HEM checkpoint to HF ===")
    try:
        api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
        api.upload_folder(
            folder_path=hem_final_dir,
            path_in_repo="checkpoint-hem-final",  # HF tag (human-readable, independent of local dirname)
            repo_id=OUTPUT_REPO,
            commit_message=f"HEM retrained: {len(HEM_HARD_EXAMPLES)} hard examples, loss={hem_final_loss}",
        )
        print(f"  ✓ HF upload OK")
    except Exception as e:
        print(f"  ⚠️  HF upload falhou: {e}")

    if AUTO_SUBMIT:
        print(f"\n  === KAGGLE SUBMIT (HEM retrained) ===")
        hem_zip = f"/tmp/kg1_submit/submission_hard_retrained.zip"
        try:
            create_submission_zip(hem_final_dir, hem_zip)
            desc = f"{CFG['name']} HEM-retrained {len(HEM_HARD_EXAMPLES)}ex loss-{hem_final_loss}"
            submit_to_kaggle(hem_zip, desc)
        except Exception as e:
            print(f"  ❌ HEM submit error: {e}")

    # ============================================================
    # 6) CLEANUP: devolve model para estado utilizável por Cell 4.5
    # ============================================================
    # Cell 4.5 (averaging) faz glob.glob nos checkpoints — nosso checkpoint-99999
    # já está lá. Modelo continua em memória com pesos HEM-retrained.
    model.config.use_cache = True
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n  === HEM RETRAIN SUMMARY ===")
    print(f"  Hard examples: {len(HEM_HARD_EXAMPLES)}")
    print(f"  Final loss: {hem_final_loss}")
    print(f"  Steps: {hem_trainer.state.global_step}")
    print(f"  Time: {hem_elapsed/3600:.2f}h")
    print(f"  Checkpoint: {hem_final_dir}")
    print(f"  (Cell 4.5 vai incluir este no averaging automaticamente)")

print("\n✅ CELL 4.7 COMPLETE")


In [ ]:
#@title 🔀 CELL 4.5: Checkpoint Averaging + Per-Family Eval (NONUPLE)
#@markdown ### AIMO-2 trick: linearly merge last 4 checkpoints for +1-2 score points free

if not CHECKPOINT_AVERAGING:
    print("⏭️  CELL 4.5 SKIPPED (Phase 1-3 backward compat)")
else:
    import glob, torch
    from safetensors.torch import load_file, save_file
    from huggingface_hub import HfApi

    print("=" * 60)
    print("  NONUPLE CHECKPOINT AVERAGING (AIMO-2 trick)")
    print("=" * 60)

    # Find all checkpoints in OUTPUT_DIR
    ckpts = sorted(
        glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
        key=lambda p: int(p.rsplit("-", 1)[-1]),
    )
    print(f"  Found {len(ckpts)} checkpoints in {OUTPUT_DIR}")

    if len(ckpts) < 2:
        print(f"  ⚠️  Need >= 2 checkpoints for averaging, found {len(ckpts)}")
        print(f"  ⏭️  Skipping averaging")
    else:
        # Use last 4 (or all if fewer)
        ckpts_to_average = ckpts[-4:]
        print(f"  Averaging {len(ckpts_to_average)} checkpoints:")
        for c in ckpts_to_average:
            print(f"    {c}")

        # Load all adapter_model.safetensors
        states = []
        for c in ckpts_to_average:
            adapter_file = f"{c}/adapter_model.safetensors"
            if not os.path.exists(adapter_file):
                print(f"  ⚠️  Skipping {c}: no adapter_model.safetensors")
                continue
            states.append(load_file(adapter_file))

        if len(states) < 2:
            print(f"  ❌ Could not load enough valid checkpoints")
        else:
            # Linear average
            avg_state = {}
            for key in states[0].keys():
                avg_state[key] = sum(s[key].float() for s in states) / len(states)
                # Restore original dtype (bfloat16)
                avg_state[key] = avg_state[key].to(states[0][key].dtype)

            # Save averaged adapter
            avg_dir = f"{OUTPUT_DIR}/averaged"
            os.makedirs(avg_dir, exist_ok=True)
            save_file(avg_state, f"{avg_dir}/adapter_model.safetensors")

            # Copy adapter_config.json from latest checkpoint
            import shutil
            shutil.copy2(f"{ckpts_to_average[-1]}/adapter_config.json", f"{avg_dir}/adapter_config.json")

            print(f"  ✓ Averaged adapter saved to {avg_dir}")

            # Upload averaged to HF
            try:
                api = HfApi()
                api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
                api.upload_folder(
                    folder_path=avg_dir,
                    path_in_repo="averaged",
                    repo_id=OUTPUT_REPO,
                    commit_message=f"AIMO-2 averaged: {len(states)} checkpoints",
                )
                print(f"  ✓ Uploaded averaged to https://huggingface.co/{OUTPUT_REPO}/tree/main/averaged")
            except Exception as e:
                print(f"  ⚠️  Upload failed: {e}")

            # Auto-submit averaged version
            if AUTO_SUBMIT:
                print(f"\n  === Submitting AVERAGED to Kaggle ===")
                avg_zip = f"/tmp/kg1_submit/submission_averaged.zip"
                try:
                    create_submission_zip(avg_dir, avg_zip)
                    desc = f"{CFG['name']} AVERAGED ({len(states)} ckpts) NONUPLE"
                    submit_to_kaggle(avg_zip, desc)
                except Exception as e:
                    print(f"  ❌ Averaged submit error: {e}")

    print("\n✅ CELL 4.5 COMPLETE")


In [ ]:
#@title 📈 CELL 5: Check Scores + Diagnostics

print("=== Checking Kaggle submissions ===")
try:
    result = subprocess.run(
        ["kaggle", "competitions", "submissions",
         "-c", COMPETITION, "--csv"],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        lines = result.stdout.strip().split("\n")
        print(f"\nRecent submissions:")
        for line in lines[:10]:
            print(f"  {line}")
    else:
        print(f"Error: {result.stderr}")
except Exception as e:
    print(f"Error checking submissions: {e}")

# Training loss curve
if trainer.state.log_history:
    print(f"\n=== Training Loss Curve ===")
    losses = [(h.get("step", 0), h.get("loss", None))
              for h in trainer.state.log_history if h.get("loss") is not None]
    for step, loss in losses:
        bar = "█" * int(max(0, 40 - loss * 5))
        print(f"  Step {step:5d}: loss={loss:.4f} {bar}")
    
    if losses:
        first_loss = losses[0][1]
        last_loss = losses[-1][1]
        print(f"\n  First loss: {first_loss:.4f}")
        print(f"  Final loss: {last_loss:.4f}")
        print(f"  Reduction: {(1-last_loss/first_loss)*100:.1f}%")
        
        if last_loss < 1.0:
            print(f"  ✅ Excelente convergência! Score esperado: 0.70+")
        elif last_loss < 2.0:
            print(f"  ✅ Boa convergência. Score esperado: 0.65-0.72")
        elif last_loss < 3.5:
            print(f"  ⚠️ Convergência mediana. Score esperado: 0.60-0.68")
        else:
            print(f"  ❌ Convergência fraca. Score esperado: < 0.60")

print("\n✅ CELL 5 COMPLETE")

In [ ]:
#@title 🔄 CELL 6: Manual Submit (checkpoint específico)
#@markdown Use esta célula para submeter um checkpoint manualmente.

CHECKPOINT_STEP = 400  #@param {type:"integer"}
SUBMIT_DESC = ""  #@param {type:"string"}

import glob as g

# Find checkpoint
ckpt_dir = f"{OUTPUT_DIR}/checkpoint-{CHECKPOINT_STEP}"
if not os.path.exists(ckpt_dir):
    # Try to find closest checkpoint
    ckpts = sorted(g.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    print(f"Available checkpoints: {[os.path.basename(c) for c in ckpts]}")
    if ckpts:
        ckpt_dir = ckpts[-1]
        print(f"Using latest: {ckpt_dir}")
    else:
        print("❌ No checkpoints found!")
        ckpt_dir = None

if ckpt_dir and os.path.exists(ckpt_dir):
    zip_path = f"/tmp/kg1_submit/manual_step{CHECKPOINT_STEP}.zip"
    create_submission_zip(ckpt_dir, zip_path)
    
    if not SUBMIT_DESC:
        SUBMIT_DESC = f"{CFG['name']} manual-step-{CHECKPOINT_STEP} r{LORA_RANK}-a{LORA_ALPHA}"
    
    submit_to_kaggle(zip_path, SUBMIT_DESC)
    print("\n✅ Manual submit complete")

---
## 📋 Instruções de Uso

### Fase 1 (v30 - SFT Perfected):
1. Configure `PHASE = 1`, `FRESH_LORA = True`
2. Execute Cells 1-4 sequencialmente
3. Aguarde os auto-submits nos steps configurados
4. Use Cell 5 para verificar scores
5. **Gate**: se score ≥ 0.70, avance para Fase 2

### Fase 2 (v31 - CoT Distillation):
1. Configure `PHASE = 2`, `FRESH_LORA = False`
2. Certifique-se que o adapter da Fase 1 está no HF
3. Execute Cells 1-4
4. **Gate**: se score ≥ 0.75, avance para Fase 3

### Fase 3 (v32 - GRPO RL):
1. Configure `PHASE = 3`, `FRESH_LORA = False`
2. ⚠️ GRPO precisa de GRPOTrainer (Cell 4 usa SFTTrainer)
3. Para GRPO, use o script dedicado `hf_train_grpo_v2.py`

### Secrets do Colab:
- `HF_KEY`: token HuggingFace
- `KAGGLE_USERNAME`: felipe1983
- `KAGGLE_KEY`: (nova key)

### Regras de Ouro:
- **NUNCA** alpha > 16
- **NUNCA** aceitar score < 0.68
- **SEMPRE** submeter múltiplos checkpoints
- Se loss no step 10 > 8.0: ABORTAR e investigar